In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

In [ ]:
# Cargar y explorar el dataset
dataset = load_dataset("rajistics/electricity_demand", split="train")
df = pd.DataFrame(dataset)
print("Primeras filas del dataset:")
print(df.head())


In [ ]:
# Preprocesamiento
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')
df['date'] = df['timestamp'].dt.date

In [ ]:
# Agrupar por día y asegurar 48 intervalos por día
daily_series = df.groupby('date')['demand'].apply(list)
daily_series = daily_series[daily_series.apply(lambda x: len(x) == 48)]

In [ ]:
# Crear dataset supervisado (día N -> día N+1)
X = []
y = []
for i in range(len(daily_series) - 1):
    X.append(daily_series.iloc[i])
    y.append(daily_series.iloc[i + 1])

X = np.array(X)
y = np.array(y)

In [ ]:
# Separar en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)


In [ ]:
# Entrenar modelo Random Forest
model = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42))
model.fit(X_train, y_train)

In [ ]:
# Predicciones
y_pred = model.predict(X_test)


In [ ]:
# Calcular métricas
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_absolute_error(y_test, y_pred, squared=False)
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

In [ ]:
# Visualización de la primera predicción
plt.figure(figsize=(12, 5))
plt.plot(y_test[0], label="Real")
plt.plot(y_pred[0], label="Predicción")
plt.title("Predicción vs Realidad para el primer día de test")
plt.xlabel("Intervalos de 30 minutos")
plt.ylabel("Demanda eléctrica")
plt.legend()
plt.grid(True)
plt.show()